In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#xii time sampled

mass = 1.0
k_const = 1.0
x_0 = 0.7
v_0 = 1.2

In [3]:
import torch

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- sampling sizes ----
N_interior = 15000
N_boundary = 1000

# ---- domain ranges ----
z_min, z_max = 0.0, 20.0
xi_min, xi_max = 0.1, 0.4


# =========================================================
# Interior collocation points
# =========================================================
z_interior = torch.rand(N_interior, 1, device=device) * (z_max - z_min) + z_min
xi_interior = torch.rand(N_interior, 1, device=device) * (xi_max - xi_min) + xi_min

# Stack inputs → shape (N, 2)
interior_points = torch.cat([z_interior, xi_interior], dim=1)


# =========================================================
# Boundary (initial condition) points  → z = 0
# =========================================================
z_boundary = torch.zeros(N_boundary, 1, device=device)
xi_boundary = torch.rand(N_boundary, 1, device=device) * (xi_max - xi_min) + xi_min

boundary_points = torch.cat([z_boundary, xi_boundary], dim=1)


# =========================================================
# Print shapes (sanity check)
# =========================================================
print("Interior points shape:", interior_points.shape)   # (2000, 2)
print("Boundary points shape:", boundary_points.shape)   # (100, 2)


Interior points shape: torch.Size([15000, 2])
Boundary points shape: torch.Size([1000, 2])


In [4]:
import torch.nn as nn
import torch.optim as optim
import math

device = "cuda" if torch.cuda.is_available() else "cpu"


In [5]:
class Sine(nn.Module):
    def __init__(self, w0=1.0):
        super().__init__()
        self.w0 = w0

    def forward(self, x):
        return torch.sin(self.w0 * x)

In [6]:
class PINN(nn.Module):
    def __init__(self, in_dim=2, width=64, depth=4, out_dim=1, w0=30.0, x0 = 0.7, v0 = 1.2):
        super().__init__()
        self.x0 = x0
        self.v0 = v0
        layers = []

        # ----- first layer -----
        layers.append(nn.Linear(in_dim, width))
        layers.append(Sine(w0))

        # ----- hidden layers -----
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(Sine(1.0))   # hidden layers use w0 = 1

        # ----- final linear layer -----
        layers.append(nn.Linear(width, out_dim))

        self.net = nn.Sequential(*layers)

        # ----- SIREN initialization -----
        self.init_weights(w0)

    # ----------------------------
    # SIREN weight initialization
    # ----------------------------
    def init_weights(self, w0):
        with torch.no_grad():
            for i, m in enumerate(self.net):
                if isinstance(m, nn.Linear):
                    in_dim = m.weight.size(1)

                    if i == 0:
                        # first layer initialization (special)
                        m.weight.uniform_(-1 / in_dim, 1 / in_dim)
                    else:
                        # hidden layers initialization
                        bound = math.sqrt(6 / in_dim) / w0
                        m.weight.uniform_(-bound, bound)

                    nn.init.zeros_(m.bias)

    # ----------------------------
    # forward pass
    # ----------------------------
    def forward(self, z, xi):
        inp = torch.cat([z, xi], dim=1)

        N = self.net(inp)          # unconstrained NN output

        # ---- hard initial-condition constraint ----
        x = self.x0 + self.v0 * z + (z ** 2) * N
        
        return x

In [7]:
def derivatives(model, z, xi):
    z = z.clone().detach().requires_grad_(True)

    x = model(z, xi)

    dx = torch.autograd.grad(
        x, z, torch.ones_like(x), create_graph=True
    )[0]

    ddx = torch.autograd.grad(
        dx, z, torch.ones_like(dx), create_graph=True
    )[0]

    return x, dx, ddx


In [8]:
def ode_loss(model, z, xi):
    x, dx, ddx = derivatives(model, z, xi)
    residual = ddx + 2 * xi * dx + x
    return torch.mean(residual**2)


In [9]:
# def ic_loss(model, z_b, xi_b, x0, v0):
#     x, dx, _ = derivatives(model, z_b, xi_b)

#     loss_x = torch.mean((x - x0) ** 2)
#     loss_v = torch.mean((dx - v0) ** 2)

#     return loss_x + loss_v


In [10]:
def total_loss(model, z_i, xi_i, z_b, xi_b, x0, v0, lam_ic=10.0):
    return ode_loss(model, z_i, xi_i)

In [11]:
z_i  = interior_points[:, 0:1].to(device)
xi_i = interior_points[:, 1:2].to(device)

z_b  = boundary_points[:, 0:1].to(device)   # should be all zeros
xi_b = boundary_points[:, 1:2].to(device)

model = PINN().to(device)

In [14]:
adam = torch.optim.Adam(model.parameters(), lr=1e-5)

for epoch in range(2000):
    adam.zero_grad()

    loss = total_loss(model, z_i, xi_i, z_b, xi_b, x_0, v_0)

    loss.backward()
    adam.step()

    if epoch % 50 == 0:
        print(f"Adam epoch {epoch} | loss = {loss.item():.3e}")


Adam epoch 0 | loss = 2.528e+00
Adam epoch 50 | loss = 2.459e+00
Adam epoch 100 | loss = 2.410e+00
Adam epoch 150 | loss = 2.365e+00
Adam epoch 200 | loss = 2.323e+00
Adam epoch 250 | loss = 2.282e+00
Adam epoch 300 | loss = 2.242e+00
Adam epoch 350 | loss = 2.203e+00
Adam epoch 400 | loss = 2.165e+00
Adam epoch 450 | loss = 2.128e+00
Adam epoch 500 | loss = 2.091e+00
Adam epoch 550 | loss = 2.055e+00
Adam epoch 600 | loss = 2.020e+00
Adam epoch 650 | loss = 1.986e+00
Adam epoch 700 | loss = 1.953e+00
Adam epoch 750 | loss = 1.920e+00
Adam epoch 800 | loss = 1.888e+00
Adam epoch 850 | loss = 1.856e+00
Adam epoch 900 | loss = 1.825e+00
Adam epoch 950 | loss = 1.795e+00
Adam epoch 1000 | loss = 1.766e+00
Adam epoch 1050 | loss = 1.738e+00
Adam epoch 1100 | loss = 1.709e+00
Adam epoch 1150 | loss = 1.685e+00
Adam epoch 1200 | loss = 1.655e+00
Adam epoch 1250 | loss = 1.629e+00
Adam epoch 1300 | loss = 1.603e+00
Adam epoch 1350 | loss = 1.579e+00
Adam epoch 1400 | loss = 1.554e+00
Adam epo

In [ ]:
lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=8.0,
    max_iter=10000,
    tolerance_grad=1e-9,
    tolerance_change=1e-12,
    history_size=50,
    line_search_fn="strong_wolfe",
)

iter_lbfgs = [0]

def closure():
    lbfgs.zero_grad()
    loss = total_loss(model, z_i, xi_i, z_b, xi_b, x_0, v_0)
    loss.backward()
    
    if iter_lbfgs[0] % 10 == 0:
        print(f"L-BFGS iter {iter_lbfgs[0]:4d} | loss = {loss.item():.3e}")
    iter_lbfgs[0] += 1
    
    return loss

lbfgs.step(closure)

print("Final loss:", closure().item())


L-BFGS iter    0 | loss = 1.495e-01
L-BFGS iter   10 | loss = 1.494e-01
L-BFGS iter   20 | loss = 1.494e-01
L-BFGS iter   30 | loss = 1.493e-01
L-BFGS iter   40 | loss = 1.492e-01
L-BFGS iter   50 | loss = 1.492e-01
L-BFGS iter   60 | loss = 1.491e-01
L-BFGS iter   70 | loss = 1.490e-01
L-BFGS iter   80 | loss = 1.489e-01
L-BFGS iter   90 | loss = 1.487e-01
L-BFGS iter  100 | loss = 1.487e-01
L-BFGS iter  110 | loss = 1.486e-01
L-BFGS iter  120 | loss = 1.486e-01
L-BFGS iter  130 | loss = 1.485e-01
L-BFGS iter  140 | loss = 1.484e-01
L-BFGS iter  150 | loss = 1.484e-01
L-BFGS iter  160 | loss = 1.483e-01
L-BFGS iter  170 | loss = 1.484e-01
L-BFGS iter  180 | loss = 1.482e-01
L-BFGS iter  190 | loss = 1.486e-01
L-BFGS iter  200 | loss = 1.488e-01
L-BFGS iter  210 | loss = 1.477e-01
L-BFGS iter  220 | loss = 1.475e-01
L-BFGS iter  230 | loss = 1.474e-01
L-BFGS iter  240 | loss = 1.477e-01
L-BFGS iter  250 | loss = 1.472e-01
L-BFGS iter  260 | loss = 1.469e-01
L-BFGS iter  270 | loss = 1.

In [ ]:
def sol_x(xii, time, x_0=0.7, v_0=1.2):
    """
    Analytic solution of dimensionless damped oscillator:
        x'' + 2ξ x' + x = 0

    Inputs:
        xii  : damping ratio tensor
        time : time tensor
        x_0  : initial position
        v_0  : initial velocity

    Returns:
        x(t, ξ) as PyTorch tensor
    """

    # ensure tensors
    xii = torch.as_tensor(xii)
    time = torch.as_tensor(time)

    # damped frequency (ω0 = 1 in normalized system)
    omega_d = torch.sqrt(1 - xii**2)

    # constants from initial conditions
    A = x_0
    B = (v_0 + xii * x_0) / omega_d

    # exponential decay
    coeff = torch.exp(-xii * time)

    # solution
    return coeff * (A * torch.cos(omega_d * time) + B * torch.sin(omega_d * time))


In [ ]:
def evaluate_model(model, sol_x, m, k, N=2000, device="cpu"):
    model.eval()

    # sample evaluation points
    t = torch.linspace(0, 20, N, device=device).unsqueeze(1)
    xi = torch.linspace(0.1, 0.4, N, device=device).unsqueeze(1)

    # make meshgrid of (t, xi)
    T, XI = torch.meshgrid(t.squeeze(), xi.squeeze(), indexing="ij")
    T = T.reshape(-1, 1)
    XI = XI.reshape(-1, 1)

    # PINN prediction
    with torch.no_grad():
        x_pred = model(T, XI)

    # true solution (must return tensor)
    x_gt = sol_x(XI, T)

    # -------- error metrics --------
    mse = torch.mean((x_pred - x_gt) ** 2)

    rel_l2 = torch.norm(x_pred - x_gt) / torch.norm(x_gt)

    max_err = torch.max(torch.abs(x_pred - x_gt))

    return {
        "MSE": mse.item(),
        "Relative L2": rel_l2.item(),
        "Max error": max_err.item(),
    }


In [ ]:
metrics = evaluate_model(model, sol_x, m=1.0, k=1.0, device=device)

for k, v in metrics.items():
    print(f"{k}: {v:.3e}")


In [ ]:
print("ODE loss:", ode_loss(model, z_i, xi_i).item())
print("IC loss:", ic_loss(model, z_b, xi_b, x_0, v_0).item())

# gradient norm per loss (simple approximate)
odel = ode_loss(model, z_i, xi_i)
g_ode = torch.autograd.grad(odel, model.parameters(), create_graph=False, retain_graph=True)
norm_ode = torch.sqrt(sum([g.detach().norm()**2 for g in g_ode]))
print("grad norm (ODE):", norm_ode.item())

icl = ic_loss(model, z_b, xi_b, x_0, v_0)
g_ic = torch.autograd.grad(icl, model.parameters(), create_graph=False, retain_graph=True)
norm_ic = torch.sqrt(sum([g.detach().norm()**2 for g in g_ic]))
print("grad norm (IC):", norm_ic.item())
